Este script se encarga de la extracción periódica de datos de mercado desde la API de CoinMarketCap
con el objetivo de construir un dataset intrasesión de alta frecuencia. A través del endpoint
`cryptocurrency/listings/latest`, se descargan las 50 principales criptomonedas por capitalización
de mercado, incluyendo información de precios, volumen y métricas asociadas, todas convertidas a USD.

La lógica de extracción se encapsula en una función que realiza la petición HTTP, procesa la respuesta
en formato JSON y normaliza los datos en un DataFrame estructurado. A cada observación se le añade
un timestamp que identifica el momento exacto de la consulta, permitiendo posteriormente el análisis
temporal intrasesión.

El script ejecuta esta función de forma iterativa a intervalos regulares de 15 minutos, almacenando
los resultados de manera incremental en un archivo CSV. En la primera iteración se crea el archivo
con cabeceras y, en las siguientes, los nuevos datos se añaden sin sobrescribir la información previa.
Este enfoque permite construir una serie temporal con múltiples observaciones por criptomoneda a lo
largo de la sesión.

Finalmente, el dataset consolidado se carga en memoria para su inspección inicial y posterior
análisis exploratorio y cuantitativo.

In [ ]:
import pandas as pd
import json
from requests import Session
from time import sleep
from time import time
import os

In [ ]:
CSV_FILE = "CoinMarket_data.csv"

def api_runner():

    url = 'https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest'
    parameters = {
    'start':'1',
    'limit':'50',
    'convert':'USD'
    }
    headers = {
    'Accepts': 'application/json',
    'X-CMC_PRO_API_KEY': 'xxxxxxxxxxxxxxx',
    }

    session = Session()
    session.headers.update(headers)

    try:
        response = session.get(url, params=parameters)
        data = json.loads(response.text)
        #print(data)
    except (ConnectionError, Timeout, TooManyRedirects) as e:
        print(e)

    df = pd.json_normalize(data['data'])
    df['timestamp'] = pd.to_datetime('now')

    return df

In [ ]:
for i in range(50):
    df_iter = api_runner()

    # Si el archivo no existe, escribe cabecera
    if not os.path.exists(CSV_FILE):
        df_iter.to_csv(CSV_FILE, index=False, mode='w')
    else:
        df_iter.to_csv(CSV_FILE, index=False, mode='a', header=False)

    print(f"Iteración {i+1} guardada en CSV")
    sleep(900)

Iteración 1 guardada en CSV
Iteración 2 guardada en CSV
Iteración 3 guardada en CSV
Iteración 4 guardada en CSV
Iteración 5 guardada en CSV
Iteración 6 guardada en CSV
Iteración 7 guardada en CSV
Iteración 8 guardada en CSV
Iteración 9 guardada en CSV
Iteración 10 guardada en CSV
Iteración 11 guardada en CSV
Iteración 12 guardada en CSV
Iteración 13 guardada en CSV


In [ ]:
df = pd.read_csv('coinmarket_data.csv')
df.head()

,id,name,symbol,slug,num_market_pairs,date_added,tags,max_supply,circulating_supply,total_supply,...,quote.USD.market_cap_dominance,quote.USD.fully_diluted_market_cap,quote.USD.tvl,quote.USD.last_updated,platform.id,platform.name,platform.symbol,platform.slug,platform.token_address,timestamp
0,1,Bitcoin,BTC,bitcoin,12560,2010-07-13T00:00:00.000Z,"['mineable', 'pow', 'sha-256', 'store-of-value...",2.100000e+07,1.998908e+07,1.998908e+07,...,58.4144,1.465596e+12,NaN,2026-02-14T09:09:00.000Z,NaN,NaN,NaN,NaN,NaN,2026-02-14 10:11:17.908928
1,1027,Ethereum,ETH,ethereum,11566,2015-08-07T00:00:00.000Z,"['pos', 'smart-contracts', 'ethereum-ecosystem...",NaN,1.206925e+08,1.206925e+08,...,10.5325,2.515354e+11,NaN,2026-02-14T09:09:00.000Z,NaN,NaN,NaN,NaN,NaN,2026-02-14 10:11:17.908928
2,825,Tether USDt,USDT,tether,171208,2015-02-25T00:00:00.000Z,"['stablecoin', 'asset-backed-stablecoin', 'usd...",NaN,1.838170e+11,1.879648e+11,...,7.6941,1.878951e+11,NaN,2026-02-14T09:09:00.000Z,1027.0,Ethereum,ETH,ethereum,0xdac17f958d2ee523a2206206994597c13d831ec7,2026-02-14 10:11:17.908928
3,52,XRP,XRP,xrp,1798,2013-08-04T00:00:00.000Z,"['medium-of-exchange', 'enterprise-solutions',...",1.000000e+11,6.091732e+10,9.998572e+10,...,3.6964,1.449124e+11,NaN,2026-02-14T09:09:00.000Z,NaN,NaN,NaN,NaN,NaN,2026-02-14 10:11:17.908928
4,1839,BNB,BNB,bnb,3134,2017-07-25T00:00:00.000Z,"['marketplace', 'centralized-exchange', 'payme...",1.363593e+08,1.363593e+08,1.363593e+08,...,3.5819,8.554172e+10,NaN,2026-02-14T09:09:00.000Z,NaN,NaN,NaN,NaN,NaN,2026-02-14 10:11:17.908928
